# 📦 Citadel Publish Contract - Testing Center

## Overview

Publish and validate **three protected AI assets** through the AI Hub Gateway using the **Citadel Publish Contract**, then confirm reachability, policy application, usage tracking, and resiliency.

| # | Asset | `assetType` | Result |
|---|---|---|---|
| 1 | Weather Tool | `mcp-from-api` | Sample `weather-api` exposed as an MCP tool server |
| 2 | Microsoft Learn Tool | `mcp-existing` | Remote MS Learn MCP server published + protected |
| 3 | HR Chat Agent | `a2a` | Foundry-hosted agent published as a native A2A endpoint |

## Expected outcomes

- Three assets published via Bicep (`az deployment sub create`)
- MCP `initialize` + `tools/list` handshakes succeed through the gateway
- The A2A agent card is retrievable at `/.well-known/agent.json` and a JSON-RPC `message/send` routes to Foundry
- `mcp-usage` / `a2a-usage` custom metrics appear in Application Insights
- Each remote MCP backend has a native circuit breaker attached
- The published **HR agent** answers an HR question via gateway-routed A2A, and a direct MCP `tools/call` returns a result

## Azure Prerequisites

- An existing Citadel Governance Hub deployment (APIM + App Insights + Cosmos + usage Logic Apps)
- The sample APIs deployed (`isMCPSampleDeployed = true`) so `weather-api` exists as the API→MCP source
- A Foundry **prompt agent** to expose via A2A (see `citadel-agent-frameworks-tests.ipynb` to create one)
- Permission to assign roles on the Foundry project (Owner / User Access Administrator) — step 3️⃣.1 grants the APIM identity **Foundry access** automatically
- Azure credentials with permission to deploy at subscription scope

> **Access Contract note:** publishing (phase 1) does not grant access. This notebook then deploys a **real Citadel Access Contract** — a single `MULTI-` product that grants **all three published asset types at once** and applies **asset-type-aware** policies (LLM token limits + model RBAC; request-based rate limits for Tools and Agents). The contract mints the `api-key` used by the validation steps below.

<a id='0'></a>
### 0️⃣ Initialize Notebook Variables

Set `init_from_azd = True` to autoload the governance hub RG/location/subscription from your active `azd` environment, or `False` to fill the `REPLACE` values manually. Provide the Foundry agent coordinates for the A2A asset.

In [ ]:
import os, sys, json, time, uuid, requests
sys.path.insert(1, '../shared')
import utils
from apimtools import APIMClientTool

# ============================================================================
# 🔧 INITIALIZATION MODE
# ============================================================================
init_from_azd = True   # Set False to fill the REPLACE values below manually.

# ============================================================================
# 🔧 GOVERNANCE HUB CONFIGURATION
# ============================================================================
governance_hub_resource_group = "REPLACE"   # RG of the deployed Citadel Governance Hub
location = "REPLACE"                         # Azure region (e.g. "swedencentral")
subscription_id = "REPLACE"                  # Subscription hosting the hub

# ============================================================================
# 🤖 FOUNDRY AGENT (A2A target) CONFIGURATION
# The A2A asset republishes an EXISTING Foundry prompt agent with incoming A2A enabled.
# ============================================================================
enable_a2a_asset      = True
foundry_account_name  = "REPLACE"   # e.g. aif-citadel-agent-08
foundry_project_name  = "REPLACE"   # e.g. proj-citadel-agent-08
foundry_agent_name    = "REPLACE"   # e.g. HR-ChatAgent

# ============================================================================
# 🔐 ACCESS CONTRACT KEY VAULT (optional)
# When enabled, the MULTI- access contract publishes the single shared api-key
# PLUS one endpoint secret per asset (LLM + Tools + Agents) to this Key Vault,
# so the contract owner has every endpoint needed to reach all granted assets.
# When disabled, the contract outputs credentials directly (existing behavior).
# ============================================================================
use_access_contract_kv  = True       # Set True to publish shared key + per-asset endpoints to Key Vault
access_contract_kv_name = "REPLACE"  # Target Key Vault name (auto-loaded from azd when available)
# External Key Vault: leave both empty to use the hub's subscription / resource group,
# or set them to target a Key Vault in a DIFFERENT subscription / resource group.
access_contract_kv_subscription_id = ""  # empty => hub subscription (subscription_id)
access_contract_kv_resource_group  = ""  # empty => hub resource group (governance_hub_resource_group)

# ============================================================================
# 🧪 TEST HARNESS
# ============================================================================
publish_deployment_name = "citadel-publish-contracts-validation"
test_product_id = "PUBLISH-CONTRACT-TEST"   # temporary product used only to mint an api-key

def _is_unset(v):
    return v is None or v == "" or v == "REPLACE"

if init_from_azd:
    utils.print_info("Loading configuration from azd environment...")
    loaded = utils.load_azd_env({
        "resource_group":  ["AZURE_RESOURCE_GROUP", "GOVERNANCE_HUB_RESOURCE_GROUP"],
        "location":        ["AZURE_LOCATION", "LOCATION"],
        "subscription_id": ["AZURE_SUBSCRIPTION_ID"],
        "key_vault_name":  ["AZURE_KEY_VAULT_NAME", "KEY_VAULT_NAME"],
        "ai_foundry_services": (["AI_FOUNDRY_SERVICES"], "json"),
    }, verbose=False)
    if _is_unset(governance_hub_resource_group) and "resource_group" in loaded:
        governance_hub_resource_group = loaded["resource_group"]
    if _is_unset(location) and "location" in loaded:
        location = loaded["location"]
    if _is_unset(subscription_id) and "subscription_id" in loaded:
        subscription_id = loaded["subscription_id"]
    if _is_unset(access_contract_kv_name) and "key_vault_name" in loaded:
        access_contract_kv_name = loaded["key_vault_name"]
    if "ai_foundry_services" in loaded and isinstance(loaded["ai_foundry_services"], list) and loaded["ai_foundry_services"]:
        first = loaded["ai_foundry_services"][0]
        if _is_unset(foundry_account_name):
            foundry_account_name = first.get("cognitiveServiceName") or first.get("name") or foundry_account_name
        if _is_unset(foundry_project_name):
            ep = first.get("foundryProjectEndpoint", "")
            if "/projects/" in ep:
                foundry_project_name = ep.rstrip("/").rsplit("/projects/", 1)[-1]

# Key Vault publishing needs a resolvable vault name; disable gracefully if missing.
if use_access_contract_kv and _is_unset(access_contract_kv_name):
    utils.print_warning("use_access_contract_kv=True but no Key Vault name resolved; set access_contract_kv_name. Falling back to direct output.")
    use_access_contract_kv = False

utils.print_ok(f"Resource group : {governance_hub_resource_group}")
utils.print_ok(f"Location       : {location}")
utils.print_ok(f"Subscription   : {subscription_id}")
utils.print_ok(f"Foundry agent  : {foundry_account_name}/{foundry_project_name}/{foundry_agent_name} (a2a={enable_a2a_asset})")
if use_access_contract_kv:
    _kv_scope = "external" if (access_contract_kv_subscription_id or access_contract_kv_resource_group) else "hub"
    utils.print_ok(f"Access-contract KV : {access_contract_kv_name} ({_kv_scope} sub={access_contract_kv_subscription_id or 'hub'} rg={access_contract_kv_resource_group or 'hub'})")
else:
    utils.print_ok("Access-contract KV : disabled (direct output)")
utils.print_ok("Notebook variables initialized!")


<a id='1'></a>
### 1️⃣ Verify Azure CLI and Connected Subscription

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")
if output.success and output.json_data:
    utils.print_info(f"Current user: {output.json_data['user']['name']}")
    utils.print_info(f"Subscription ID: {output.json_data['id']}")
    if _is_unset(subscription_id):
        subscription_id = output.json_data['id']
    if subscription_id != output.json_data['id']:
        utils.print_warning(f"Active subscription differs from configured ({subscription_id}). Run: az account set --subscription {subscription_id}")

<a id='2'></a>
### 2️⃣ Initialize APIM Client Tool

Discover the deployed APIM instance and its gateway URL.

In [ ]:
apimClientTool = APIMClientTool(governance_hub_resource_group)
apimClientTool.initialize()
apim_name = apimClientTool.apim_resource_name
gateway_url = str(apimClientTool.apim_resource_gateway_url)
utils.print_ok(f"APIM: {apim_name}")
utils.print_ok(f"Gateway URL: {gateway_url}")

<a id='3'></a>
### 3️⃣ Enable incoming A2A on the Foundry agent

Foundry prompt agents support the responses protocol, but the **A2A endpoint must be activated** with a `PATCH` that sets the agent card and enables the `a2a` protocol. This mirrors [`local/contracts/a2a-foundry-sample-agent.ps1`](../local/contracts/a2a-foundry-sample-agent.ps1) and the Learn guide: [Enable incoming A2A on a Foundry agent](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/enable-agent-to-agent-endpoint).

> Requires the **Foundry User** role on the project. Skipped automatically when `enable_a2a_asset = False`.

In [ ]:
a2a_card_backend_url = ""
a2a_jsonrpc_backend_url = ""
if enable_a2a_asset and not (_is_unset(foundry_account_name) or _is_unset(foundry_project_name) or _is_unset(foundry_agent_name)):
    base_url = f"https://{foundry_account_name}.services.ai.azure.com/api/projects/{foundry_project_name}"
    a2a_card_backend_url = f"{base_url}/agents/{foundry_agent_name}/endpoint/protocols/a2a/agentCard/v1.0"
    a2a_jsonrpc_backend_url = f"{base_url}/agents/{foundry_agent_name}/endpoint/protocols/a2a"
    token = utils.run("az account get-access-token --resource https://ai.azure.com --query accessToken -o tsv").text.strip()
    body = {
        "agent_card": {
            "description": "HR Chat Agent published via the AI Hub Gateway (A2A).",
            "version": "1.0",
            "skills": [{"id": "general-qa", "name": "General Q&A", "description": "Answers HR policy questions"}]
        },
        "agent_endpoint": {"protocol_configuration": {"responses": {}, "a2a": {}}}
    }
    r = requests.patch(f"{base_url}/agents/{foundry_agent_name}?api-version=v1",
                       headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
                       data=json.dumps(body), timeout=60)
    if r.status_code < 300:
        utils.print_ok(f"Incoming A2A enabled on {foundry_agent_name}")
        utils.print_info(f"Agent card backend: {a2a_card_backend_url}")
    else:
        utils.print_error(f"A2A enable failed ({r.status_code}): {r.text[:400]}")
else:
    utils.print_warning("A2A asset disabled or Foundry agent not fully specified; skipping A2A enablement.")

<a id='3-1'></a>
### 3️⃣.1 Grant the APIM managed identity access on the Foundry project

The published A2A backend authenticates to Foundry with **APIM's user-assigned managed identity** (audience `https://ai.azure.com`) — no keys. For that to work the identity needs a data-plane role on the Foundry project.

This step:
1. Discovers the APIM identity (prefers a **user-assigned** identity; falls back to system-assigned).
2. Grants it the Foundry role on the target project scope.
3. Captures the UAMI **`clientId`**, which is passed to the deployment as `managedIdentityClientId` so the backend embeds MI auth using that specific identity.

> Default role is `Azure AI User`. For least privilege you can switch `foundry_role` to `Foundry Agent Consumer`. Requires Owner / User Access Administrator on the project. Skipped when `enable_a2a_asset = False`.

In [ ]:
# Identity passed to the deployment ('' => APIM system-assigned identity)
apim_mi_client_id = ""
apim_mi_principal_id = ""
foundry_role = "Foundry Agent Consumer"   # least-privilege alternative: "Foundry Agent Consumer"

if enable_a2a_asset and not _is_unset(foundry_account_name):
    # 1) APIM identity — prefer a user-assigned identity, else system-assigned
    idj = utils.run(f"az apim show -g {governance_hub_resource_group} -n {apim_name} --query identity -o json",
                    "Retrieved APIM identity", "Failed to read APIM identity")
    if idj.success and idj.json_data:
        ident = idj.json_data
        uami = ident.get("userAssignedIdentities") or {}
        if uami:
            _rid, _val = next(iter(uami.items()))
            apim_mi_client_id = _val.get("clientId", "")
            apim_mi_principal_id = _val.get("principalId", "")
            utils.print_info(f"APIM user-assigned identity: clientId={apim_mi_client_id}")
        elif ident.get("principalId"):
            apim_mi_principal_id = ident["principalId"]
            utils.print_info("APIM system-assigned identity (managedIdentityClientId left empty)")

    # 2) Foundry project resource id (account may live in a different resource group)
    foundry_project_id = ""
    acc = utils.run(f"az cognitiveservices account list --query \"[?name=='{foundry_account_name}'].id\" -o tsv")
    account_id = acc.text.strip() if acc.success else ""
    if account_id:
        foundry_project_id = f"{account_id}/projects/{foundry_project_name}"

    # 3) Grant the role so APIM's identity can call the Foundry agent
    if apim_mi_principal_id and foundry_project_id:
        ra = utils.run(
            f"az role assignment create --assignee-object-id {apim_mi_principal_id} "
            f"--assignee-principal-type ServicePrincipal --role \"{foundry_role}\" --scope {foundry_project_id}",
            f"Granted '{foundry_role}' to APIM identity on the Foundry project",
            "Role assignment failed (may already exist, or you lack Owner/User Access Administrator)")
        if not ra.success:
            utils.print_warning("If the role name is unavailable in your tenant, try 'Foundry Agent Consumer' or assign it in the portal.")
    else:
        utils.print_warning("Could not resolve APIM principalId or Foundry project id; grant the Foundry role manually.")
else:
    utils.print_info("A2A asset disabled; skipping Foundry access grant.")

<a id='3-2'></a>
### 3️⃣.2 Ensure the sample source API exists (`weather-api`) — subscription-protected

The Weather Tool is published with `assetType = mcp-from-api`, which turns an **existing** APIM API into an MCP server. That source API (`weather-api`) is part of the main accelerator but is only deployed when `isMCPSampleDeployed = true`, so it may be missing on your gateway — which makes the publish contract deployment fail.

This step creates `weather-api` on demand (idempotent) from the same spec and mock policy shipped in the accelerator ([`openapi.json`](../bicep/infra/modules/apim/sample/weather/openapi.json) + [`policy.xml`](../bicep/infra/modules/apim/sample/weather/policy.xml)), including its `get-weather` operation that the publish contract references.

> 🔐 **Secure source API (key forwarding).** The source API is created **subscription-protected** and reads its key from a **custom header** (`x-mcp-sub-key`). An `mcp-from-api` server forwards `tools/call` to the source API *internally*; APIM strips the standard subscription headers (`api-key` / `Ocp-Apim-Subscription-Key`) before that hop, but leaves a non-standard header intact. The Weather Tool is then published with `forwardSubscriptionKeyToSource: true` (step 4️⃣), so the MCP policy injects the caller's contract key into `x-mcp-sub-key`. Result: the **same contract key** reaches both the MCP tool and the raw API, and **direct calls without a key return `401`** (no anonymous access). The access contract (step 5️⃣) adds `weather-api` to the product so that one key authorizes both.


In [ ]:
# Ensure the sample 'weather-api' exists — it is the source API for the mcp-from-api Weather Tool.
# The main accelerator only deploys it when isMCPSampleDeployed=true, so create it on demand here.
from azure.mgmt.apimanagement.models import (
    ApiCreateOrUpdateParameter, PolicyContract, SubscriptionKeyParameterNamesContract)

_client = apimClientTool.client
_rg, _svc = governance_hub_resource_group, apim_name
weather_api_id = "weather-api"
# Custom (non-standard) header the source API reads its key from. Standard subscription headers
# (api-key / Ocp-Apim-Subscription-Key) are stripped by APIM before the internal tools/call hop,
# so a custom name is required for the forwarded key to survive. Reused by steps 4️⃣ + 5️⃣.
weather_source_key_header = "x-mcp-sub-key"

_spec_path = "../bicep/infra/modules/apim/sample/weather/openapi.json"
_policy_path = "../bicep/infra/modules/apim/sample/weather/policy.xml"
with open(_spec_path, "r", encoding="utf-8") as f:
    _spec = f.read()
with open(_policy_path, "r", encoding="utf-8") as f:
    _policy = f.read()

# Upsert (idempotent). SECURE approach: keep the source API subscription-PROTECTED but read the key
# from the custom header the MCP policy forwards, so there is no anonymous direct access to /weather.
utils.print_info(f"Ensuring source API '{weather_api_id}' (subscription-protected, custom header {weather_source_key_header})...")
_client.api.begin_create_or_update(_rg, _svc, weather_api_id, ApiCreateOrUpdateParameter(
    path="weather",
    display_name="Weather API",
    description="Weather API for getting dynamic weather information for a given location.",
    format="openapi+json",
    value=_spec,
    protocols=["https"],
    subscription_required=True,
    subscription_key_parameter_names=SubscriptionKeyParameterNamesContract(
        header=weather_source_key_header, query=weather_source_key_header),
    service_url="https://to-be-replaced-by-policy",
)).result()
# Mock response policy — returns synthetic weather data, so no real backend is called.
_client.api_policy.create_or_update(_rg, _svc, weather_api_id, "policy",
                                    PolicyContract(value=_policy, format="rawxml"))
utils.print_ok(f"Source API '{weather_api_id}' ready (subscription required, key header {weather_source_key_header}).")

# Confirm the operation the publish contract references is present.
_ops = [o.name for o in _client.api_operation.list_by_api(_rg, _svc, weather_api_id)]
(utils.print_ok if "get-weather" in _ops else utils.print_error)(f"weather-api operations: {_ops}")


<a id='4'></a>
### 4️⃣ Publish the assets (deploy the Publish Contract)

Generate a `.bicepparam` under the source-control folder `citadel-publish-contracts/contracts/sample-assets/dev/` and deploy the `citadel-publish-contracts` module at subscription scope.

In [ ]:
assets = [
    {
        "assetType": "mcp-from-api", "name": "weather-tool", "displayName": "Weather Tool (MCP)",
        "description": "Weather data operations, published as an MCP tool server.", "path": "weather-tool-mcp",
        "metadata": {"version": "1.0.0", "owner": "Platform Engineering", "classification": "internal"},
        "sourceApiName": "weather-api", "operationNames": ["get-weather"],
        # Secure: forward the caller's contract key to the subscription-protected source API (step 3️⃣.2)
        # via a custom header APIM won't strip, so the same key reaches the tool AND the raw API.
        "forwardSubscriptionKeyToSource": True,
        "sourceSubscriptionKeyHeaderName": globals().get("weather_source_key_header", "x-mcp-sub-key"),
        "publishToApiCenter": False,
    },
    {
        "assetType": "mcp-existing", "name": "ms-learn-tool", "displayName": "Microsoft Learn Tool (MCP)",
        "description": "Microsoft Learn MCP server published through the gateway.", "path": "ms-learn-tool-mcp",
        "transportType": "streamable", "subscriptionRequired": True,
        "metadata": {"version": "1.0.0", "owner": "Knowledge Mgmt", "classification": "public"},
        "backend": {"url": "https://learn.microsoft.com/api/mcp", "authType": "none"}, "publishToApiCenter": False,
    },
]
if enable_a2a_asset and a2a_card_backend_url:
    assets.append({
        "assetType": "a2a", "name": "hr-chat-agent", "displayName": "HR Chat Agent (A2A)",
        "description": "HR assistant published via A2A.", "path": "hr-chat-agent",
        "agentId": foundry_agent_name, "subscriptionRequired": True, "subscriptionKeyHeaderName": "api-key", "agentCardPath": "/.well-known/agent.json",
        "agentCardBackendUrl": a2a_card_backend_url, "jsonRpcPath": "/",
        "metadata": {"version": "1.0.0", "owner": "HR Digital", "classification": "confidential"},
        "backend": {"url": a2a_jsonrpc_backend_url, "authType": "managed-identity", "authConfig": {"resource": "https://ai.azure.com"}},
        "publishToApiCenter": False,
    })

def _bicep(v, ind=0):
    pad = "  " * ind
    if isinstance(v, bool):
        return "true" if v else "false"
    if isinstance(v, (int, float)):
        return str(v)
    if isinstance(v, str):
        return "'" + v.replace("'", "\\'") + "'"
    if isinstance(v, list):
        return "[\n" + "".join(f"{pad}  {_bicep(i, ind+1)}\n" for i in v) + f"{pad}]"
    if isinstance(v, dict):
        return "{\n" + "".join(f"{pad}  {k}: {_bicep(val, ind+1)}\n" for k, val in v.items()) + f"{pad}}}"
    return "''"

param_text = (
    "using '../../../main.bicep'\n\n"
    f"param apim = {{\n  subscriptionId: '{subscription_id}'\n  resourceGroupName: '{governance_hub_resource_group}'\n  name: '{apim_name}'\n}}\n\n"
    # APIM user-assigned identity clientId so the A2A backend authenticates to Foundry as that UAMI
    f"param managedIdentityClientId = '{globals().get('apim_mi_client_id', '')}'\n"
    "param configureCircuitBreaker = true\n"
    # Prefix tool endpoints under /mcp and agent endpoints under /agent (default). Set False for legacy paths.
    "param useAssetTypePathPrefix = true\n\n"
    f"param publishAssets = {_bicep(assets)}\n"
)
# Source-control folder: contracts/<publish-contract>/<env>/ (mirrors the access-contract layout)
publish_bicep_dir = "../bicep/infra/citadel-publish-contracts"
publish_contract_name, publish_env = "sample-assets", "dev"
publish_contract_dir = os.path.join(publish_bicep_dir, "contracts", publish_contract_name, publish_env)
os.makedirs(publish_contract_dir, exist_ok=True)
param_path = os.path.join(publish_contract_dir, "main.bicepparam")
with open(param_path, "w", encoding="utf-8") as f:
    f.write(param_text)
utils.print_info(f"Wrote {param_path} with {len(assets)} assets")
print(param_text)

In [ ]:
cmd = (
    f"az deployment sub create --name {publish_deployment_name} --location {location} "
    f"--template-file {os.path.join(publish_bicep_dir, 'main.bicep')} "
    f"--parameters {param_path} -o json"
)
out = utils.run(cmd, "Publish contract deployed", "Publish contract deployment failed", print_output=False)
if out.success and out.json_data:
    published = out.json_data.get("properties", {}).get("outputs", {}).get("publishedAssets", {}).get("value", [])
    for a in published:
        utils.print_ok(f"{a['assetType']:<13} {a['name']:<16} -> {a['endpoint']}")
    globals()['published_assets'] = published
    # Name -> deployed asset (authoritative endpoint/path incl. the mcp//agent/ prefix) for the validation cells.
    globals()['published_by_name'] = {a['name']: a for a in published}

<a id='5'></a>
### 5️⃣ Grant access with a real Access Contract (mixed asset types)

Instead of a throwaway product, deploy a **real Citadel Access Contract** that grants this consumer access to **all three published asset types at once** — the LLM inference APIs, the published **Tools (MCP)**, and the published **Agent (A2A)** — under a **single `MULTI-` product**. The contract is written to `citadel-access-contracts/contracts/<biz>-<usecase>/<env>/`, following the same source-control layout.

The contract's product policy classifies each request with the `set-asset-kind` fragment and applies **asset-type-specific controls**:

| Asset kind | Controls |
| --- | --- |
| **LLM** | model RBAC (`validate-model-access`) + `llm-token-limit` (TPM + quota) |
| **Tool (MCP)** | request-based `rate-limit-by-key` (calls/min) + `quota-by-key` |
| **Agent (A2A)** | request-based `rate-limit-by-key` (calls/min) + `quota-by-key` |

The deployment mints the subscription **api-key** used by the validation steps below. Backward compatible: existing `LLM-` contracts are untouched.

> 🔐 **Optional Key Vault (set `use_access_contract_kv = True` in step 0️⃣):** the contract shares **one** api-key across every asset and publishes it plus **one endpoint secret per asset** to Key Vault. Every endpoint uses the **default auto-generated secret name** (`<code>-<bu>-<useCase>-<env>-<apiName>-endpoint`) so distinct LLM front-doors never collide — e.g. `universal-llm-api` resolves to `/models` and `azure-openai-api` to `/openai` in their own secrets. `foundryApiName` marks the LLM endpoint (`universal-llm-api`, `/models`) for Foundry connections.
>
> **External Key Vault:** set `access_contract_kv_subscription_id` / `access_contract_kv_resource_group` in step 0️⃣ to publish to a Key Vault in a **different subscription / resource group** (the deploying identity needs *Key Vault Secrets Officer* there). Leave both empty to use the hub's Key Vault. Step 5️⃣.1 verifies the secrets landed.


In [ ]:
import os, json, time

# Real Access Contract that GRANTS access to the published assets (replaces the old throwaway product).
client = apimClientTool.client
rg, svc = governance_hub_resource_group, apim_name

# --- Classify the published assets + discover which LLM inference APIs exist on this gateway ---
tool_apis  = [a["name"] for a in assets if a["assetType"] in ("mcp-from-api", "mcp-existing")]
agent_apis = [a["name"] for a in assets if a["assetType"] == "a2a"]
_candidate_llm = ["universal-llm-api", "azure-openai-api", "unified-ai-api"]
existing_apis = {a.name for a in client.api.list_by_service(rg, svc)}
llm_apis = [n for n in _candidate_llm if n in existing_apis]
granted_apis = llm_apis + tool_apis + agent_apis
# Source APIs behind forwarding mcp-from-api tools must join the SAME product so the shared key
# authorizes both the MCP tool and the protected raw API (forwardSubscriptionKeyToSource).
source_apis = list(dict.fromkeys(
    a["sourceApiName"] for a in assets
    if a["assetType"] == "mcp-from-api" and a.get("forwardSubscriptionKeyToSource") and a.get("sourceApiName") in existing_apis))
product_apis = granted_apis + [n for n in source_apis if n not in granted_apis]
utils.print_info(f"Contract grants -> LLM:{llm_apis}  Tools:{tool_apis}  Agents:{agent_apis}  Sources:{source_apis}")

# --- Contract identity: MULTI- prefix because it mixes asset types (single-type would be LLM-/TOOL-/AGENT-) ---
biz_unit, use_case_name, env = "Governance", "PublishedAssets", "DEV"
_types_present = (1 if llm_apis else 0) + (1 if tool_apis else 0) + (1 if agent_apis else 0)
contract_code = "MULTI" if _types_present > 1 else ("AGENT" if agent_apis else "TOOL" if tool_apis else "LLM")
product_id = f"{contract_code}-{biz_unit}-{use_case_name}-{env}"
sub_name = f"{product_id}-SUB-01"
test_product_id, sub_id = product_id, sub_name  # reused by the cleanup cell

# --- Per-asset-type limits (kept low for Tools/Agents so the burst tests below trip a 429) ---
TOOL_CALLS_PER_MIN, AGENT_CALLS_PER_MIN = 20, 10
allowed_models_csv = "gpt-4.1,gpt-5.4-mini"

# --- Generate the asset-type-aware product policy (conditional on set-asset-kind) ---
policy_xml = f'''<policies>
    <inbound>
        <base />
        <!-- COMMON POLICIES (asset-agnostic): add opt-in content safety / custom alerting here. -->
        <set-variable name="contractToolApis" value="{",".join(tool_apis)}" />
        <set-variable name="contractAgentApis" value="{",".join(agent_apis)}" />
        <include-fragment fragment-id="set-asset-kind" />
        <choose>
            <when condition="@(context.Variables.GetValueOrDefault<string>("assetKind","llm") == "llm")">
                <include-fragment fragment-id="set-llm-requested-model" />
                <set-variable name="allowedModels" value="{allowed_models_csv}" />
                <include-fragment fragment-id="validate-model-access" />
                <llm-token-limit counter-key="@(context.Subscription.Id)" tokens-per-minute="10000" estimate-prompt-tokens="false" token-quota="1000000" token-quota-period="Monthly" />
                <set-variable name="enableResponseHeaders" value="@(true)" />
            </when>
            <when condition="@(context.Variables.GetValueOrDefault<string>("assetKind","") == "tool")">
                <rate-limit-by-key calls="{TOOL_CALLS_PER_MIN}" renewal-period="60" counter-key="@(context.Subscription.Id + ":tool")" />
                <quota-by-key calls="100000" renewal-period="2592000" counter-key="@(context.Subscription.Id + ":tool")" />
            </when>
            <when condition="@(context.Variables.GetValueOrDefault<string>("assetKind","") == "agent")">
                <rate-limit-by-key calls="{AGENT_CALLS_PER_MIN}" renewal-period="60" counter-key="@(context.Subscription.Id + ":agent")" />
                <quota-by-key calls="50000" renewal-period="2592000" counter-key="@(context.Subscription.Id + ":agent")" />
            </when>
        </choose>
    </inbound>
    <backend><base /></backend>
    <outbound><base /></outbound>
    <on-error><base /></on-error>
</policies>'''

# Source-control folder: contracts/<biz>-<usecase>/<env>/ (mirrors the access-contract notebook)
access_bicep_dir = "../bicep/infra/citadel-access-contracts"
contract_dir = os.path.join(access_bicep_dir, "contracts", f"{biz_unit.lower()}-{use_case_name.lower()}", env.lower())
os.makedirs(contract_dir, exist_ok=True)
with open(os.path.join(contract_dir, "ai-product-policy.xml"), "w", encoding="utf-8") as f:
    f.write(policy_xml)

# --- Multi-asset endpoint publishing: ONE shared key + ONE endpoint secret per granted asset ---
# Every asset (each LLM API, each Tool, each Agent) uses the DEFAULT auto-generated secret name
# (<code>-<bu>-<usecase>-<env>-<apiName>-endpoint). Giving distinct LLM APIs the SAME name would
# collide in Key Vault (last write wins) and could surface e.g. azure-openai-api's /openai instead
# of universal-llm-api's /models. Foundry is wired to universal-llm-api via foundryApiName below.
asset_endpoints = [{"apiName": _n} for _n in granted_apis]
foundry_api_name = "universal-llm-api" if "universal-llm-api" in llm_apis else (llm_apis[0] if llm_apis else granted_apis[0])

def _bicep_asset_endpoints(items):
    lines = []
    for it in items:
        if it.get("endpointSecretName"):
            lines.append(f"      {{ apiName: '{it['apiName']}', endpointSecretName: '{it['endpointSecretName']}' }}")
        else:
            lines.append(f"      {{ apiName: '{it['apiName']}' }}")
    return "[\n" + "\n".join(lines) + "\n    ]"

# External Key Vault support: default to the hub sub/rg, override for a Key Vault in another subscription.
kv_sub_param = access_contract_kv_subscription_id or subscription_id
kv_rg_param  = access_contract_kv_resource_group or rg
kv_name_param = access_contract_kv_name if use_access_contract_kv else "unused-kv"
use_kv_param = "true" if use_access_contract_kv else "false"

# Product association covers the consumer endpoints PLUS any forwarded source APIs (so one key authorizes both).
api_list_bicep = "[" + ", ".join("'" + n + "'" for n in product_apis) + "]"
param_text = f'''using '../../../main.bicep'

param apim = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{rg}'
  name: '{svc}'
}}
param keyVault = {{
  subscriptionId: '{kv_sub_param}'
  resourceGroupName: '{kv_rg_param}'
  name: '{kv_name_param}'
}}
param useTargetAzureKeyVault = {use_kv_param}
param useCase = {{
  businessUnit: '{biz_unit}'
  useCaseName: '{use_case_name}'
  environment: '{env}'
}}
param apiNameMapping = {{
  {contract_code}: {api_list_bicep}
}}
param services = [
  {{
    code: '{contract_code}'
    apiKeySecretName: 'PUBLISHED-ASSETS-KEY'
    foundryApiName: '{foundry_api_name}'
    assetEndpoints: {_bicep_asset_endpoints(asset_endpoints)}
    policyXml: loadTextContent('ai-product-policy.xml')
  }}
]
param productTerms = 'Citadel Access Contract (mixed asset types) for publish-contract validation'
param useTargetFoundry = false
'''
param_path = os.path.join(contract_dir, "main.bicepparam")
with open(param_path, "w", encoding="utf-8") as f:
    f.write(param_text)
utils.print_info(f"Wrote access contract -> product {product_id} ({len(product_apis)} APIs, {len(asset_endpoints)} endpoint secrets, KV={'on' if use_access_contract_kv else 'off'})")

deploy_name = f"publish-access-contract-{time.strftime('%H%M%S')}"
cmd = f"az deployment sub create --name {deploy_name} --location {location} --template-file {os.path.join(access_bicep_dir, 'main.bicep')} --parameters {param_path} -o json"
out = utils.run(cmd, "Access contract deployed", "Access contract deployment failed", print_output=False)

api_key = None
# Capture per-asset Key Vault secret names (populated when use_access_contract_kv=True) for the verify cell.
access_contract_key_secret_name = None
access_contract_endpoint_secret_names = []
if out.success and out.json_data:
    outputs = out.json_data.get("properties", {}).get("outputs", {})
    eps = outputs.get("endpoints", {}).get("value", [])
    if eps:
        api_key = eps[0].get("apiKey")
    subs = outputs.get("subscriptions", {}).get("value", [])
    if subs:
        access_contract_key_secret_name = subs[0].get("keyVaultApiKeySecretName") or None
        access_contract_endpoint_secret_names = subs[0].get("keyVaultEndpointSecretNames", []) or []
if not api_key:
    secrets = client.subscription.list_secrets(rg, svc, sub_name)
    api_key = secrets.primary_key
utils.print_ok(f"Access-contract api-key acquired (product {product_id})")
if use_access_contract_kv:
    utils.print_ok(f"Key Vault '{access_contract_kv_name}' (sub={kv_sub_param}, rg={kv_rg_param}): key secret '{access_contract_key_secret_name}' + {len(access_contract_endpoint_secret_names)} endpoint secrets")

In [ ]:
# --- (Optional) Verify the shared key + per-asset endpoints landed in Key Vault ---
# Only runs when use_access_contract_kv=True. Confirms one key secret + one endpoint secret per asset.
_res = globals().setdefault('results', {})
if use_access_contract_kv and not _is_unset(access_contract_kv_name):
    utils.print_info(f"Reading access-contract secrets from Key Vault '{access_contract_kv_name}'...")
    if access_contract_key_secret_name:
        k = utils.run(f"az keyvault secret show --vault-name {access_contract_kv_name} --name {access_contract_key_secret_name} --query value -o tsv", print_output=False)
        utils.print_ok(f"key      {access_contract_key_secret_name}: {'present' if (k.success and k.text.strip()) else 'MISSING'}")
    for name in access_contract_endpoint_secret_names:
        e = utils.run(f"az keyvault secret show --vault-name {access_contract_kv_name} --name {name} --query value -o tsv", print_output=False)
        val = e.text.strip() if e.success else ""
        (utils.print_ok if val else utils.print_error)(f"endpoint {name}: {val or 'MISSING'}")
    _res['access-contract-kv-secrets'] = bool(access_contract_endpoint_secret_names) and all(
        (utils.run(f"az keyvault secret show --vault-name {access_contract_kv_name} --name {n} --query value -o tsv", print_output=False).text or "").strip()
        for n in access_contract_endpoint_secret_names
    )
else:
    utils.print_info("Access-contract Key Vault publishing disabled; skipping KV secret verification.")


<a id='6'></a>
### 6️⃣ Validate the MCP tools (handshake + tools/list)

Perform the MCP `initialize` handshake and list tools over the streamable HTTP transport for both published tool servers.

> **Endpoint convention:** with `useAssetTypePathPrefix = true` (default) every **tool** is served under an `mcp/` prefix and every **agent** under an `agent/` prefix. For an **API→MCP** tool (`mcp-from-api`) APIM also appends `/mcp`, so the endpoint is `{gateway}/mcp/{path}/mcp`. For a **native/remote MCP** server (`mcp-existing`) the endpoint is `{gateway}/mcp/{path}` (no trailing `/mcp`). The cells below prefer the authoritative `endpoint` returned by the deployment (`publishedAssets`) and fall back to computing the prefixed path. Set the toggle to `false` (or a per-asset `pathPrefix: ''`) to keep legacy un-prefixed paths.


In [ ]:
def mcp_call(endpoint, api_key, method, params=None, mcp_session=None):
    headers = {"Content-Type": "application/json", "Accept": "application/json, text/event-stream", "api-key": api_key}
    if mcp_session:
        headers["Mcp-Session-Id"] = mcp_session
    # APIM's MCP runtime requires a NUMERIC JSON-RPC id; a non-numeric string (e.g. a UUID) is rejected as 'Invalid JSON payload'.
    mcp_call._id = getattr(mcp_call, "_id", 0) + 1
    payload = {"jsonrpc": "2.0", "id": mcp_call._id, "method": method}
    if params is not None:
        payload["params"] = params
    r = requests.post(endpoint, headers=headers, data=json.dumps(payload), timeout=60)
    text, data = r.text, None
    if "text/event-stream" in r.headers.get("content-type", ""):
        for line in text.splitlines():
            if line.startswith("data:"):
                try:
                    data = json.loads(line[5:].strip()); break
                except Exception:
                    pass
    else:
        try:
            data = r.json()
        except Exception:
            pass
    return r, data

def validate_mcp(endpoint, api_key, label):
    utils.print_info(f"--- {label}: {endpoint}")
    r, data = mcp_call(endpoint, api_key, "initialize", {
        "protocolVersion": "2025-06-18", "capabilities": {},
        "clientInfo": {"name": "citadel-validation", "version": "1.0"}})
    session = r.headers.get("Mcp-Session-Id")
    if r.status_code < 300 and data and "result" in data:
        utils.print_ok(f"{label}: initialize OK (session={session})")
        r2, data2 = mcp_call(endpoint, api_key, "tools/list", {}, mcp_session=session)
        tools = (data2 or {}).get("result", {}).get("tools", []) if data2 else []
        utils.print_ok(f"{label}: tools/list -> {[t.get('name') for t in tools]}")
        return True
    utils.print_error(f"{label}: initialize failed ({r.status_code}) {r.text[:300]}")
    return False

results = {}

# Default asset-type path prefix, mirroring the publish contract (Tools -> mcp/, Agents -> agent/).
_ASSET_TYPE_PREFIX = {"mcp-from-api": "mcp", "mcp-existing": "mcp", "a2a": "agent"}

def _prefixed_path(asset):
    prefix = _ASSET_TYPE_PREFIX.get(asset["assetType"], "")
    return f"{prefix}/{asset['path']}" if prefix else asset["path"]

def mcp_endpoint(asset):
    # Prefer the authoritative endpoint returned by the deployment; fall back to the prefixed path.
    dep = globals().get("published_by_name", {}).get(asset["name"])
    if dep and dep.get("endpoint"):
        return dep["endpoint"]
    base = f"{gateway_url}/{_prefixed_path(asset)}"
    return f"{base}/mcp" if asset["assetType"] == "mcp-from-api" else base

def agent_endpoint(asset):
    # Prefer the deployed (prefixed) path; fall back to the mirrored agent/ prefix.
    dep = globals().get("published_by_name", {}).get(asset["name"])
    if dep and dep.get("path"):
        return f"{gateway_url}/{dep['path']}"
    return f"{gateway_url}/{_prefixed_path(asset)}"

for a in assets:
    if a['assetType'] in ('mcp-from-api', 'mcp-existing'):
        results[a['name']] = validate_mcp(mcp_endpoint(a), api_key, a['displayName'])

<a id='7'></a>
### 7️⃣ Validate the A2A agent (agent card + JSON-RPC)

Fetch the agent card that APIM re-exposes at `/.well-known/agent.json`, then send a JSON-RPC `message/send` that the gateway proxies (with its managed identity) to the Foundry agent.

In [ ]:
if enable_a2a_asset and a2a_card_backend_url:
    hr_agent_asset = next(a for a in assets if a["assetType"] == "a2a")
    agent_base = agent_endpoint(hr_agent_asset)
    card_url = f"{agent_base}/.well-known/agent.json"
    utils.print_info(f"Validating A2A agent card at {card_url}...")
    # The A2A API is subscriptionRequired; present the test api-key in the configured header.
    rc = requests.get(card_url, headers={"api-key": api_key}, timeout=60)
    if rc.status_code < 300:
        utils.print_ok(f"Agent card reachable (HTTP {rc.status_code}) via {card_url}")
        results['hr-chat-agent-card'] = True
    else:
        utils.print_error(f"Agent card fetch failed ({rc.status_code}): {rc.text[:300]}")
        results['hr-chat-agent-card'] = False

    # Foundry serves A2A v0.3 by default; the message object requires a 'kind' field. Numeric JSON-RPC id.
    rpc = {"jsonrpc": "2.0", "id": 1, "method": "message/send",
           "params": {"message": {"kind": "message", "role": "user", "messageId": str(uuid.uuid4()),
                                   "parts": [{"kind": "text", "text": "What can you help me with?"}]}}}
    rr = requests.post(agent_base, headers={"Content-Type": "application/json", "api-key": api_key},
                       data=json.dumps(rpc), timeout=120)
    if rr.status_code < 300:
        utils.print_ok(f"A2A message/send routed to Foundry (HTTP {rr.status_code})")
        results['hr-chat-agent-rpc'] = True
    else:
        utils.print_warning(f"A2A message/send returned {rr.status_code}: {rr.text[:300]}")
        results['hr-chat-agent-rpc'] = False
else:
    utils.print_warning("A2A asset not published; skipping agent validation.")

<a id='8'></a>
### 8️⃣ Validate usage tracking (custom metrics)

The baseline policies emit `McpRequests` (namespace `mcp-usage`) and `A2ARequests` (namespace `a2a-usage`) on each inbound call. Query Application Insights `customMetrics` to confirm they land (metrics may take a couple of minutes to appear).

In [ ]:
app_insights_name = None
ai_out = utils.run(f"az resource list -g {governance_hub_resource_group} --resource-type Microsoft.Insights/components -o json")
if ai_out.success and ai_out.json_data:
    # Prefer the APIM app insights (name usually contains 'apim')
    comps = ai_out.json_data
    app_insights_name = next((c['name'] for c in comps if 'apim' in c['name'].lower()), comps[0]['name'])
    utils.print_info(f"App Insights: {app_insights_name}")
    kql = "customMetrics | where name in ('McpRequests','A2ARequests') | summarize count=sum(valueSum) by name, tostring(customDimensions['deploymentName']) | order by name asc"
    q = utils.run(f"az monitor app-insights query --app {app_insights_name} -g {governance_hub_resource_group} --analytics-query \"{kql}\" -o json")
    if q.success and q.json_data:
        rows = q.json_data.get('tables', [{}])[0].get('rows', [])
        if rows:
            for row in rows:
                utils.print_ok(f"metric {row[0]} | asset {row[1]} | count {row[2]}")
        else:
            utils.print_warning("No mcp-usage/a2a-usage metrics yet — allow a few minutes after the calls and re-run this cell.")
else:
    utils.print_warning("Could not locate an Application Insights component in the hub RG.")

<a id='9'></a>
### 9️⃣ Validate resiliency (circuit breaker on published backends)

Confirm the remote MCP and A2A assets got a native circuit breaker on their backend.

In [ ]:
for name in [a['name'] for a in assets if a['assetType'] == 'mcp-existing']:
    backend_id = f"{name}-backend"
    uri = f"{apimClientTool.apim_service_id}/backends/{backend_id}?api-version=2024-06-01-preview"
    b = utils.run(f"az rest --method get --uri {uri} -o json")
    if b.success and b.json_data:
        cb = b.json_data.get('properties', {}).get('circuitBreaker')
        if cb and cb.get('rules'):
            rule = cb['rules'][0]
            fc = rule.get('failureCondition', {})
            utils.print_ok(f"{backend_id}: circuit breaker ON (count={fc.get('count')}, interval={fc.get('interval')}, trip={rule.get('tripDuration')})")
        else:
            utils.print_warning(f"{backend_id}: no circuit breaker configured")
    else:
        utils.print_error(f"{backend_id}: backend not found")

<a id='10'></a>
### 🔟 Consume the published assets (Microsoft Agent Framework + direct MCP)

Prove the published assets are usable by a real client, **through the gateway**:
1. A **Microsoft Agent Framework** `A2AAgent` resolves the published **HR agent** card and answers an HR question.
2. A **direct MCP `tools/call`** invokes a published tool through the gateway.

> The publish contract **rewrites the agent card's transport URLs to the gateway**, so the A2A client routes **through the gateway** (presenting the subscription `api-key`) instead of calling Foundry directly. Requires `agent-framework` + `agent-framework-a2a` (version-aligned: `pip install -U agent-framework agent-framework-a2a`).

In [ ]:
# Microsoft Agent Framework: consume the published HR agent (A2A) through the gateway.
# The published agent card advertises the gateway's transport URLs (rewritten by the publish contract),
# so A2AAgent routes through the gateway (presenting the subscription api-key) rather than calling Foundry directly.
import nest_asyncio, asyncio, httpx
from a2a.client import A2ACardResolver
from agent_framework.a2a import A2AAgent
nest_asyncio.apply()

if enable_a2a_asset and a2a_card_backend_url:
    agent_url = agent_endpoint(next(a for a in assets if a["assetType"] == "a2a"))

    async def ask_hr_agent(question):
        # A2A API is subscriptionRequired; present the test api-key on every gateway call (card + JSON-RPC).
        async with httpx.AsyncClient(timeout=120.0, headers={"api-key": api_key}) as http_client:
            resolver = A2ACardResolver(httpx_client=http_client, base_url=agent_url)
            card = await resolver.get_agent_card(relative_card_path="/.well-known/agent.json")
            agent = A2AAgent(name=card.name, description=card.description, agent_card=card, http_client=http_client)
            resp = await agent.run(question)
            return "\n".join(getattr(m, "text", "") for m in resp.messages)

    try:
        utils.print_info("Asking the HR agent (Microsoft Agent Framework -> A2A -> gateway)...")
        answer = asyncio.run(ask_hr_agent("What is our leave policy?"))
        utils.print_ok("HR agent answered:")
        print(answer)
        results['hr-chat-agent-maf'] = bool(answer.strip())
    except Exception as e:
        utils.print_error(f"Agent Framework A2A invocation failed: {e}")
        utils.print_warning("Ensure 'agent-framework' + 'agent-framework-a2a' are installed and version-aligned "
                            "(pip install -U agent-framework agent-framework-a2a).")
        results['hr-chat-agent-maf'] = False
else:
    utils.print_warning("A2A asset not published; skipping Agent Framework A2A consumption.")

In [ ]:
# Direct invocation of a published MCP tool (initialize -> tools/call) through the gateway.
def call_mcp_tool(asset, tool_name, arguments):
    ep = mcp_endpoint(asset)
    r, _ = mcp_call(ep, api_key, "initialize", {"protocolVersion": "2025-06-18", "capabilities": {},
                    "clientInfo": {"name": "citadel-validation", "version": "1.0"}})
    session = r.headers.get("Mcp-Session-Id")
    _, data = mcp_call(ep, api_key, "tools/call", {"name": tool_name, "arguments": arguments}, mcp_session=session)
    return data

weather_asset = next((a for a in assets if a["name"] == "weather-tool"), None)
if weather_asset:
    out = call_mcp_tool(weather_asset, "get-weather", {"city": "London"})
    content = (out or {}).get("result", {}).get("content", [])
    text = content[0].get("text", "") if content else json.dumps(out)
    utils.print_ok("Direct MCP tools/call  weather-tool.get-weather('London') ->")
    print(text)
    results['weather-tool-toolcall'] = bool(content)
else:
    utils.print_warning("weather-tool not in assets; skipping direct MCP tool invocation.")

<a id='11'></a>
### 1️⃣1️⃣ Validate conditional access-contract policies (per asset type)

The `MULTI-` product applies **different** throttling per asset type via the `set-asset-kind` branch. Burst-call a **Tool** and the **Agent** to confirm the request-based `rate-limit-by-key` trips (**HTTP 429**) — proving the product policy branched on asset kind and applied the Tool/Agent controls (not the LLM token limit). LLM token limits are validated best-effort only when a model is reachable.

In [ ]:
import concurrent.futures

def _burst(url, headers, body, n):
    def _one(_):
        try:
            return requests.post(url, headers=headers, data=body, timeout=30).status_code
        except Exception:
            return -1
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as ex:
        return list(ex.map(_one, range(n)))

# --- Tool: burst past the tool rate limit -> expect 429s (throttled calls never reach the backend) ---
tool_asset = next((a for a in assets if a["assetType"] in ("mcp-from-api", "mcp-existing")), None)
if tool_asset:
    ep = mcp_endpoint(tool_asset)
    hdr = {"Content-Type": "application/json", "Accept": "application/json, text/event-stream", "api-key": api_key}
    body = json.dumps({"jsonrpc": "2.0", "id": 1, "method": "initialize",
                       "params": {"protocolVersion": "2025-06-18", "capabilities": {},
                                  "clientInfo": {"name": "burst", "version": "1.0"}}})
    codes = _burst(ep, hdr, body, TOOL_CALLS_PER_MIN + 15)
    throttled = sum(1 for c in codes if c == 429)
    (utils.print_ok if throttled else utils.print_warning)(
        f"Tool rate-limit: {throttled}/{len(codes)} calls got 429 (limit {TOOL_CALLS_PER_MIN}/min) -> tool branch applied")
    results['tool-rate-limit-429'] = throttled > 0
else:
    utils.print_warning("No tool asset; skipping tool rate-limit check.")

# --- Agent: burst past the agent rate limit -> expect 429s ---
if enable_a2a_asset and a2a_card_backend_url:
    agent_base = agent_endpoint(next(a for a in assets if a["assetType"] == "a2a"))
    hdr = {"Content-Type": "application/json", "api-key": api_key, "A2A-Version": "1.0"}
    body = json.dumps({"jsonrpc": "2.0", "id": 1, "method": "message/send",
                       "params": {"message": {"kind": "message", "role": "user", "messageId": "burst",
                                              "parts": [{"kind": "text", "text": "ping"}]}}})
    codes = _burst(agent_base, hdr, body, AGENT_CALLS_PER_MIN + 15)
    throttled = sum(1 for c in codes if c == 429)
    (utils.print_ok if throttled else utils.print_warning)(
        f"Agent rate-limit: {throttled}/{len(codes)} calls got 429 (limit {AGENT_CALLS_PER_MIN}/min) -> agent branch applied")
    results['agent-rate-limit-429'] = throttled > 0
else:
    utils.print_warning("A2A asset not published; skipping agent rate-limit check.")

<a id='results'></a>
### 📊 Results Summary

In [ ]:
utils.print_info("=== Publish Contract Validation Summary ===")
for k, v in results.items():
    (utils.print_ok if v else utils.print_error)(f"{k}: {'PASS' if v else 'FAIL'}")
if all(results.values()):
    utils.print_ok("All published assets validated successfully.")
else:
    utils.print_warning("Some checks did not pass — review the sections above.")

<a id='cleanup'></a>
### 🧹 Cleanup (Optional)

Remove the temporary test product/subscription. Set `delete_published_assets = True` to also remove the published APIs/backends.

In [ ]:
delete_published_assets = False
delete_access_contract = False
try:
    if delete_access_contract:
        client.subscription.delete(rg, svc, sub_id, if_match="*")
        client.product.delete(rg, svc, test_product_id, if_match="*", delete_subscriptions=True)
        utils.print_ok(f"Removed access-contract product {test_product_id} + subscription")
    else:
        utils.print_info(f"Access contract product {test_product_id} + subscription retained (delete_access_contract={delete_access_contract})")
except Exception as e:
    utils.print_warning(f"Cleanup of access contract failed: {e}")

if delete_published_assets:
    for a in assets:
        try:
            client.api.delete(rg, svc, a['name'], if_match="*")
            utils.print_info(f"Deleted API {a['name']}")
        except Exception as e:
            utils.print_warning(f"Could not delete API {a['name']}: {e}")
        if a['assetType'] in ('mcp-existing', 'a2a'):
            try:
                client.backend.delete(rg, svc, f"{a['name']}-backend", if_match="*")
            except Exception:
                pass
    utils.print_ok("Published assets removed")
else:
    utils.print_info(f"Published assets retained (delete_published_assets={delete_published_assets})")
